# Moonshot study on Colab

Reproduces the 2015-2026 pre-registered run end to end: fetch, screen,
assemble, train, evaluate against `docs/PREREGISTRATION_2015.md`.

## Read this before choosing a runtime

**Pick a high-CPU runtime, not a GPU one.** This is the opposite of the usual
advice and it is worth understanding why, because choosing a T4 here makes the
notebook *slower*.

| stage | what it is | scales with |
|---|---|---|
| fetch | HTTP against SEC and Yahoo | rate limits, not hardware |
| assemble | pandas merges over ~6M rows | single-core pandas, then RAM |
| train | 30 LightGBM fits (2 models x 5 folds x 3 seeds) | **CPU cores** |
| simulate | Python loop over selected trades | single core |

Nothing in that table wants a GPU.

LightGBM's GPU support accelerates *histogram construction*, which scales with
(features x bins). This panel has **80 features**. That kernel is small enough
that per-iteration host-to-device transfer dominates it, and on narrow data GPU
LightGBM is commonly slower than CPU. It also requires a source build with
`-DUSE_CUDA=1`; `pip install lightgbm` gives you the CPU build whether or not a
GPU is attached.

Meanwhile Colab's free tier is **2 vCPU**. Feature assembly took 12 minutes on
a 4-core machine and would roughly double there. So:

* **Free tier (2 vCPU):** slower than a modest laptop. Works, but expect ~2x.
* **Colab Pro high-RAM (8+ vCPU):** the right choice. Training is 30 independent
  fits, so it is close to linear in cores.
* **GPU runtime:** no benefit, and you give up CPU quota to get it.

## Memory

The assembled panel is ~1.3 GB at float32 and the merges that build it peak
several times higher. A 12 GB runtime is enough; the first attempt at this run
was OOM-killed at float64, which is why the assembler downcasts.

## Why this re-fetches instead of loading a prepared file

The prepared inputs are ~610 MB, which is past what can be pushed through a
notebook-friendly transfer. Re-fetching is about an hour on one machine -- SEC
bulk archives are 62 seconds of it -- and everything lands in the HTTP cache on
Drive, so you pay it once.

## 1. Install and mount

In [ ]:
!pip -q install git+https://github.com/nikku03/integratedai@claude/trading-ml-model-design-p5ux05
!pip -q install lightgbm pyarrow pandas_market_calendars

import multiprocessing, os
print(f"cores: {multiprocessing.cpu_count()}")
!free -g | sed -n 2p

from google.colab import drive
drive.mount('/content/drive')

# IAI_HOME is the single root -- cache/ and store/ are derived from it. Put it
# on Drive so a runtime disconnect costs minutes rather than the whole fetch.
# It must be set BEFORE importing iai: the default is resolved at import time,
# so setting it afterwards silently writes to the ephemeral container instead.
os.environ['IAI_HOME'] = '/content/drive/MyDrive/iai_home'

OUT   = '/content/drive/MyDrive/iai_2015'        # staged fetch outputs
STORE = f"{os.environ['IAI_HOME']}/store"        # what the analysis reads
UA    = 'YOUR NAME your@email.com'               # <-- REQUIRED

os.makedirs(OUT, exist_ok=True)
assert '@' in UA, 'set a real contact address: the SEC blocks the whole IP otherwise'
os.environ['IAI_USER_AGENT'] = UA

from iai.core.config import Config
_c = Config.moonshot(); _c.ensure_dirs()
print('cache:', _c.data.cache_dir)
print('store:', _c.data.store_dir)
assert 'drive' in str(_c.data.store_dir), \
    'IAI_HOME was set too late. Runtime > Restart, then run this cell first.'

## 2. Fetch

Six stages. Only `prices` and `events` shard usefully -- `insiders` downloads 45
whole-market archives, so running it on N machines costs N times the bytes for
none of the speed.

To shard, run the `prices` cell in N notebooks with `--shard i --n-shards N`,
wait for all of them, then run `screen` once.

In [ ]:
SHARD, N_SHARDS = 0, 1   # set per notebook if sharding the price stage
ARGS = f'--start 2015-01-01 --end 2026-01-01 --out {OUT} --user-agent "{UA}"'

!python -m scripts.colab_fetch --stage candidates {ARGS}

In [ ]:
# Slowest stage: ~46 min for 5,490 candidates at 2 req/s on one machine.
# Yahoo answers abuse with blocks rather than 429s, so do not raise --yahoo-rate.
!python -m scripts.colab_fetch --stage prices {ARGS} \
    --shard {SHARD} --n-shards {N_SHARDS} --yahoo-rate 2.0 --workers 6

In [ ]:
# Point-in-time cut to <=2,000 names per quarter, on trailing cap and trailing
# dollar volume. Refuses to run on a partial shard set rather than silently
# screening a different universe.
!python -m scripts.colab_fetch --stage screen {ARGS} --max-names 2000

In [ ]:
!python -m scripts.colab_fetch --stage events   {ARGS} --shard {SHARD} --n-shards {N_SHARDS} --workers 8
!python -m scripts.colab_fetch --stage insiders {ARGS}          # ONE machine only
!python -m scripts.colab_fetch --stage merge    {ARGS} --prefix w2015

!cp {OUT}/w2015_prices.parquet {OUT}/w2015_events.parquet {STORE}/
!ls -la {STORE}

## 3. Sanity checks before spending an hour on training

Cheap, and each one has caught a real bug in this pipeline.

In [ ]:
import pandas as pd

px = pd.read_parquet(f'{STORE}/w2015_prices.parquet', columns=['date','ticker','tradable'])
ev = pd.read_parquet(f'{STORE}/w2015_events.parquet',
                     columns=['source','kind','ticker','event_ts','available_ts'])

print(f"prices  {len(px):,} bars, {px.ticker.nunique():,} names, "
      f"{px.date.min().date()} .. {px.date.max().date()}")
print(f"events  {len(ev):,}\n")
print(ev.source.value_counts().to_string())

# The one that must never fail: nothing may be actionable before it happened.
assert (ev.available_ts >= ev.event_ts).all(), 'LOOKAHEAD: available_ts precedes event_ts'
print('\nPIT check passed: no event is actionable before it occurred')

# Insider events resolved by CIK, not by the filer's typed symbol. Matching on
# the symbol silently dropped 18-30% of open-market buys, zeroed per issuer and
# concentrated on companies that had been renamed.
ins = ev[ev.source == 'insiders']
print(f"insider events {len(ins):,} across {ins.ticker.nunique():,} names")

## 4. Run the pre-registered test

`--members` is not optional. Without it every name in the file is treated as
enterable on every date, including names that only qualified for the universe
years later -- which is lookahead, and the script will warn but still run.

Expect ~15 min of feature assembly then ~30-60 min of training on 4 cores,
less in proportion to how many cores you have.

In [ ]:
!python scripts/moonshot.py \
    --prefix w2015 \
    --members {OUT}/members.parquet \
    --trades-per-week 5 --target 0.10 --stop 0.07 --horizon 10 \
    --max-year-share 0.35 \
    --prereg docs/PREREGISTRATION_2015.md \
    2>&1 | tee {OUT}/moonshot_result.log

## 5. Reading the verdict

The criteria are fixed in `docs/PREREGISTRATION_2015.md` and were committed
before this data existed. The script evaluates them itself so the result cannot
be graded after the fact.

**Primary:** net return per trade with a **week-clustered** standard error,
t > 2.0. The clustering matters -- five trades in one week share a regime and
lose together, so dividing by sqrt(n) treats them as five independent draws and
inflates t. The naive t is printed alongside so the size of that dependence is
visible rather than assumed.

**Secondary:** volatility control (skill, not a volatility tilt), outlier
robustness (survives dropping the 20 largest winners), temporal spread (no year
holds >35% of trades), and the EV-vs-P(spike) ablation.

**The stopping rule is part of the pre-registration.** If the primary fails,
the answer is that this edge is not there -- not that the next catalyst class
should be tried. Searching over catalyst classes after a failure is how a
wanted result gets manufactured, and two independent samples already point the
same way.

In [ ]:
print(open(f'{OUT}/moonshot_result.log').read()[-3000:])